<a href="https://colab.research.google.com/github/zelal-Eizaldeen/deeplearning_course/blob/main/tensorflow_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- In this programming example, we will show how to **build a neural machine translation network using the transformer architecture in TensorFlow**.

This example is very similar to the neural machine translation using LSTM, but instead of using an LSTM-based encoder and decoder, we are using a transformer architecture.

We import TransformerEncoder and TransformerDecoder modules from the Keras NLP module.



In [16]:
import keras_nlp
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.text \
    import text_to_word_sequence
from tensorflow.keras.preprocessing.sequence \
    import pad_sequences
from keras_nlp.layers import TransformerEncoder
from keras_nlp.layers import TransformerDecoder
import tensorflow as tf
import logging
import math
import numpy as np
import random
tf.get_logger().setLevel(logging.ERROR)

And then the constants we are defining here

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Constants
EPOCHS = 20
BATCH_SIZE = 128
MAX_WORDS = 10000
READ_LINES = 60000
NUM_HEADS = 8
LAYER_SIZE = 256
EMBEDDING_WIDTH = 128
TEST_PERCENT = 0.2
SAMPLE_SIZE = 20
OOV_WORD = 'UNK'
PAD_INDEX = 0
OOV_INDEX = 1
START_INDEX = MAX_WORDS - 2
STOP_INDEX = MAX_WORDS - 1
MAX_LENGTH = 60
SRC_DEST_FILE_NAME = '/content/drive/MyDrive/MastersDegree/Youtube/DeepLearning/1.Perceptron/Lesson6NeuralLanguageModels/data/ara-eng/ara.txt'

Reading the training data, and tokenizing this. It's all about using the same data. It's only the model we are going to change.

In [19]:
# Function to read file.
def read_file_combined(file_name, max_len):
    file = open(file_name, 'r', encoding='utf-8')
    src_word_sequences = []
    dest_word_sequences = []
    for i, line in enumerate(file):
        if i == READ_LINES:
            break
        pair = line.split('\t')
        word_sequence = text_to_word_sequence(pair[1])
        src_word_sequence = word_sequence[0:max_len]
        src_word_sequences.append(src_word_sequence)
        word_sequence = text_to_word_sequence(pair[0])
        dest_word_sequence = word_sequence[0:max_len]
        dest_word_sequences.append(dest_word_sequence)
    file.close()
    return src_word_sequences, dest_word_sequences

In [20]:
# Functions to tokenize and un-tokenize sequences.
def tokenize(sequences):
    # "MAX_WORDS-2" used to reserve two indices
    # for START and STOP.
    tokenizer = Tokenizer(num_words=MAX_WORDS-2,
                          oov_token=OOV_WORD)
    tokenizer.fit_on_texts(sequences)
    token_sequences = tokenizer.texts_to_sequences(sequences)
    return tokenizer, token_sequences

def tokens_to_words(tokenizer, seq):
    word_seq = []
    for index in seq:
        if index == PAD_INDEX:
            word_seq.append('PAD')
        elif index == OOV_INDEX:
            word_seq.append(OOV_WORD)
        elif index == START_INDEX:
            word_seq.append('START')
        elif index == STOP_INDEX:
            word_seq.append('STOP')
        else:
            word_seq.append(tokenizer.sequences_to_texts(
                [[index]])[0])
    print(word_seq)

In [21]:
# Read file and tokenize.
src_seq, dest_seq = read_file_combined(SRC_DEST_FILE_NAME,
                                       MAX_LENGTH)
src_tokenizer, src_token_seq = tokenize(src_seq)
dest_tokenizer, dest_token_seq = tokenize(dest_seq)

In [22]:
# Prepare training data.
dest_target_token_seq = [x + [STOP_INDEX] for x in dest_token_seq]
dest_input_token_seq = [[START_INDEX] + x for x in
                        dest_target_token_seq]
src_input_data = pad_sequences(src_token_seq)
dest_input_data = pad_sequences(dest_input_token_seq,
                                padding='post')
dest_target_data = pad_sequences(
    dest_target_token_seq, padding='post', maxlen
    = len(dest_input_data[0]))

In [23]:
# Split into training and test set.
rows = len(src_input_data[:,0])
all_indices = list(range(rows))
test_rows = int(rows * TEST_PERCENT)
test_indices = random.sample(all_indices, test_rows)
train_indices = [x for x in all_indices if x not in test_indices]

train_src_input_data = src_input_data[train_indices]
train_dest_input_data = dest_input_data[train_indices]
train_dest_target_data = dest_target_data[train_indices]

test_src_input_data = src_input_data[test_indices]
test_dest_input_data = dest_input_data[test_indices]
test_dest_target_data = dest_target_data[test_indices]

# Create a sample of the test set that we will inspect in detail.
test_indices = list(range(test_rows))
sample_indices = random.sample(test_indices, SAMPLE_SIZE)
sample_input_data = test_src_input_data[sample_indices]
sample_target_data = test_dest_target_data[sample_indices]

So for the transformer, as we talked about, we need this kind of **positional encoding** on **the inputs to describe to the model in what order the input words are provided**.

we have this positional encoding that we add to the embedding. And the way we've implemented that here is that we are creating a positional embedding layer that is extending the normal embedding layer. So when we instantiate that class, **we compute the positional encoding for all positions using the formulas that we had described with sine and cosine.**

And then **when this layer is called, we get the embedding from the embedding layer and then we scale that by the square root of the embedding width**. That's a trick that has shown to be good to make the model learn better.

Then, we're taking the **embedding and then we are adding the positional encoding to it**.

In [24]:
class PositionalEmbedding(Embedding):
    def __init__(self, max_len, *args, **kwargs):
        super(PositionalEmbedding, self).__init__(*args, **kwargs)
        self.max_len = max_len
        self.positional_encodings = self.create_positional_encodings()

    def create_positional_encodings(self):
        i_range = np.arange(self.output_dim).reshape(1, self.output_dim)
        pos_range = np.arange(self.max_len).reshape(self.max_len, 1)
        sine_matrix = np.sin(1 / np.power(10000, i_range/self.output_dim) * pos_range)
        cosine_matrix = np.cos(1 / np.power(10000, (i_range-1)/self.output_dim) * pos_range)
        pos_matrix = np.zeros((self.max_len, self.output_dim))
        for i in range(self.output_dim):
            if (i % 2 == 0):
                pos_matrix[:, i] = sine_matrix[:, i]
            else:
                pos_matrix[:, i] = cosine_matrix[:, i]
        pos_matrix = pos_matrix.reshape(1, self.max_len, self.output_dim)
        return tf.cast(pos_matrix, dtype=tf.float32)

    def call(self, inputs):
        embeddings = super(PositionalEmbedding, self).call(inputs)
        embeddings = embeddings * math.sqrt(EMBEDDING_WIDTH)
        length = tf.shape(inputs)[1]
        pos_encodings = self.positional_encodings[:, :length, :]
        return embeddings + pos_encodings

    def compute_mask(self, inputs, mask=None):
        return mask

 And we can then **build the model**. And we can see here that the encoder model now consists **of the positional embedding layer and then two transformer encoder modules on top of each other**.

In [25]:
# Build encoder model.
# Input is input sequence in source language.
enc_embedding_input = Input(shape=(None, ))

# Create the encoder layers.
enc_embedding_layer = PositionalEmbedding(max_len=MAX_LENGTH, input_dim=MAX_WORDS,
                                          output_dim=EMBEDDING_WIDTH, mask_zero=True)
enc_layer1 = TransformerEncoder(intermediate_dim=LAYER_SIZE, num_heads=NUM_HEADS,
                                dropout=0.1)
enc_layer2 = TransformerEncoder(intermediate_dim=LAYER_SIZE, num_heads=NUM_HEADS,
                                dropout=0.1)

# Connect the encoder layers.
enc_embedding_layer_outputs = \
    enc_embedding_layer(enc_embedding_input)


enc_layer1_outputs = enc_layer1(enc_embedding_layer_outputs)
enc_layer2_outputs = enc_layer2(enc_layer1_outputs)

# Build the model.
enc_model = Model(enc_embedding_input, enc_layer2_outputs)
enc_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding_3          │ (None, None, 128)      │     1,280,000 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_4           │ (None, None, 128)      │       132,480 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_5           │ (None, None, 128)      │       132,480 │
│ (TransformerEncoder)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,544,960 (5.89 MB)

 Trainable params: 1,544,960 (5.89 MB)

 Non-trainable params: 0 (0.00 B)

And then on the decoder side, **we have a positional embedding layer, and then we have two transformer decoder modules on top of each other**. **And then we end with a fully connected soft max layer to predict the next word**.

In [26]:
# Build decoder model.
# Input to the network is input sequence in destination
# language and state from encoder.
dec_state_input = Input(shape=(None, EMBEDDING_WIDTH),)
dec_embedding_input = Input(shape=(None,))

# Create the encoder layers.
dec_embedding_layer = PositionalEmbedding(max_len=MAX_LENGTH, input_dim=MAX_WORDS,
                                          output_dim=EMBEDDING_WIDTH, mask_zero=True)
dec_layer1 = TransformerDecoder(intermediate_dim=LAYER_SIZE, num_heads=NUM_HEADS,
                                dropout=0.1)
dec_layer2 = TransformerDecoder(intermediate_dim=LAYER_SIZE, num_heads=NUM_HEADS,
                                dropout=0.1)
dec_layer3 = Dense(MAX_WORDS, activation='softmax')



And we can** see here that these decoder layers**, they take as inputs, not only **the embedding layer outputs, but it also takes the output from the encoder.** And that's where it can do **cross attention on that**.
for example,** the second decoder layer**. It **takes the output from the first decoder layer**, **and then it also takes this decoder state input, which it's coming from the encoder.**


In [27]:
# Connect the decoder layers.
dec_embedding_layer_outputs = dec_embedding_layer(
    dec_embedding_input)

dec_layer1_outputs = dec_layer1(dec_embedding_layer_outputs,
                                dec_state_input)
dec_layer2_outputs = dec_layer2(dec_layer1_outputs,
                                dec_state_input)
dec_layer3_outputs = dec_layer3(dec_layer2_outputs)

# Build the model.
dec_model = Model([dec_embedding_input,
                   dec_state_input],
                   dec_layer3_outputs)
dec_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │  1,280,000 │ input_layer_8[0]… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, None, 128) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decode… │ (None, None, 128) │    198,784 │ positional_embed… │
│ (TransformerDecode… │                   │            │ input_layer_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decode… │ (None, None, 128) │    198,784 │ transformer_deco… │
│ (TransformerDecode… │                   │            │ input_layer_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │  1,290,000 │ transformer_deco… │
│                     │ 10000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,967,568 (11.32 MB)

 Trainable params: 2,967,568 (11.32 MB)

 Non-trainable params: 0 (0.00 B)

 we can then put together **the encoder and the decoder into a full model, an encoder-decoder model**.

In [28]:
# Build and compile full training model.
train_enc_embedding_input = Input(shape=(None, ))
train_dec_embedding_input = Input(shape=(None, ))
intermediate_state = enc_model(train_enc_embedding_input)
train_dec_output = dec_model([train_dec_embedding_input,
                             intermediate_state])
training_model = Model([train_enc_embedding_input,
                        train_dec_embedding_input],
                        train_dec_output)
optimizer = RMSprop(learning_rate=0.001)
training_model.compile(loss='sparse_categorical_crossentropy',
                       optimizer=optimizer, metrics =['accuracy'])
training_model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_10      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_4        │ (None, None, 128) │  1,544,960 │ input_layer_9[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_5        │ (None, None,      │  2,967,568 │ input_layer_10[0… │
│ (Functional)        │ 10000)            │            │ functional_4[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,512,528 (17.21 MB)

 Trainable params: 4,512,528 (17.21 MB)

 Non-trainable params: 0 (0.00 B)

And once we have that, we can now go ahead and train this network.

we have this **for loop where we are calling fit function**,  And then after each such call, we will then loop through some samples to see the results of the translation.

In [29]:
# Train and test repeatedly.
for i in range(EPOCHS):
    print('step: ' , i)
    # Train model for one epoch.
    history = training_model.fit(
        [train_src_input_data, train_dest_input_data],
        train_dest_target_data, validation_data=(
            [test_src_input_data, test_dest_input_data],
            test_dest_target_data), batch_size=BATCH_SIZE,
        epochs=1)

    # Loop through samples to see result
    for (test_input, test_target) in zip(sample_input_data,
                                         sample_target_data):
        # Run a single sentence through encoder model.
        x = np.reshape(test_input, (1, -1))
        intermediate_states = enc_model.predict(
            x, verbose=0)
        # Provide resulting state and START_INDEX as input
        # to decoder model.
        x = np.array([[START_INDEX]])
        produced_string = ''
        pred_seq = []

        for j in range(MAX_LENGTH):
            # Predict next word and capture internal state.
            preds = dec_model.predict(
                [x, intermediate_states], verbose=0)
            # Find the most probable word.
            word_index = np.asarray(preds[0][j]).argmax()
            pred_seq.append(word_index)
            if word_index == STOP_INDEX:
                break
            x = np.append(x, [[word_index]], axis=1)
        tokens_to_words(src_tokenizer, test_input)
        tokens_to_words(dest_tokenizer, test_target)
        tokens_to_words(dest_tokenizer, pred_seq)
        print('\n\n')

step:  0
79/79 ━━━━━━━━━━━━━━━━━━━━ 374s 5s/step - accuracy: 0.7688 - loss: 5.0238 - val_accuracy: 0.8509 - val_loss: 1.0854
['PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'لنذهب', 'بالحافلة']
["let's", 'go', 'by', 'bus', 'STOP', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
['i', 'you', 'STOP']



['PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'أين', 'أختك؟']
["where's", 'your', 'sister', 'STOP', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'P

 With a transformer, we start with a **START token and do one iteration.** **That will predict the next word**. **And then we will concatenate these together.**
 We are appending the **predicted word to the input sentence.** And then we're going to next time around, and I'll provide it with both the START token as well as the first word. And that's going to **predict our second word.** And then append that.
 So now we have an **input sequence of three words, and then we do that over and over**. So we are basically **predicting the full sequence each time around, rather than doing it just one word at a time**.

We will do this for 20 times.
**We do the first epic, and that's going to then produce one set of translations**.
And then will run another 19 epics. And then we got the results

